In [1]:
import os
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras.layers import AveragePooling2D, Dense, Flatten
import time
from tqdm import tqdm
import glob
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

import numpy as np
import matplotlib.pyplot as plt


2025-02-07 02:20:12.284496: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-07 02:20:13.201172: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2025-02-07 02:20:13.201510: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory
2025-02-07 02:20:13.201517: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Cannot dlopen some TensorRT libraries. If you would like to use Nv

In [2]:
DEVICE = '/gpu:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'


In [11]:
# def get_complete_transform(output_shape, s=1.0):
#     """
#     Color distortion transform

#     Args:
#         s: Strength parameter

#     Returns:
#         A color distortion transform
#     """
# #     rnd_crop = A.RandomCrop(width=224, height=224)
# #     rnd_flip = A.HorizontalFlip(p=0.5)

# #     color_jitter = A.ColorJitter(brightness=0.8*s, contrast=0.8*s, saturation=0.8*s, hue=0.2*s, p=0.8)

# #     rnd_gray = A.ToGray(p=0.2)
# #     gaussian_blur = A.GaussianBlur(blur_limit=(3, 7), p=0.5)
# #     to_tensor = ToTensorV2()
#     image_transform =  
#     return image_transform


class ContrastiveLearningViewGenerator(object):
    """
    Take 2 random crops of 1 image as the query and key.
    """
    def __init__(self, n_views=2):
        self.base_transform = A.Compose([
        A.RandomCrop(width=224, height=224),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.2, p=0.8),
        # A.ToGray(p=0.2),
        A.GaussianBlur(blur_limit=(3, 7), p=0.5),
        ToTensorV2(),
    ])
        self.n_views = n_views

    def __call__(self, image):
        views = [self.base_transform(image=image)['image'] for i in range(self.n_views)]
        return views


In [38]:

class CustomDataset(tf.keras.utils.Sequence):
    def __init__(self, list_images, transform=None):
        self.list_images = list_images
        self.transform = A.Compose([
        A.RandomCrop(width=224, height=224),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.2, p=0.8),
        # A.ToGray(p=0.2),
        A.GaussianBlur(blur_limit=(3, 7), p=0.5),
        ToTensorV2(),
    ])

    def __len__(self):
        return len(self.list_images)

    def __getitem__(self, idx):
        if tf.is_tensor(idx):
            idx = idx.numpy()
        img_name = self.list_images[idx]
        image = cv2.imread(img_name)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)['image']
            image = tf.convert_to_tensor(image, dtype=tf.uint8)

        image = tf.expand_dims(image, axis=0)
        image = tf.transpose(image, perm=[0, 3, 2, 1])
        
        return image

In [39]:

#dataset
out_shape = [224, 224]
kernel_size = [21, 21] # 10% of out_shape

# Custom transform
# base_transforms = get_complete_transform(output_shape=out_shape)
custom_transform = ContrastiveLearningViewGenerator()

# Define the path to the directory containing the .tif files
base_path1 = "/rsrch5/home/trans_mol_path/cercan/data/segmentation/wholeDataset_inference/241119/MDA1_dataset_rois/tiles"
base_path2 = "/rsrch5/home/trans_mol_path/cercan/data/segmentation/wholeDataset_inference/241119/MDA2_dataset_rois/tiles"

# Use glob to find all .tif files in all subfolders
tif_files1 = glob.glob(f"{base_path1}/**/*.png", recursive=True)
tif_files2 = glob.glob(f"{base_path2}/**/*.png", recursive=True)
tif_files = tif_files1 + tif_files2

# Create the CustomDataset with the list of .tif files and the custom transform
deneme_ds = CustomDataset(
    list_images=tif_files
)

BATCH_SZ = 64


In [40]:

# SimCLR

foundation_model_base = '/rsrch5/home/trans_mol_path/cercan/foundationModels/physionet.org/files/medical-ai-research-foundation/1.0.0/'
def foundationModel(arch = 'path-50x1-remedis-s',no_linear=False, noAverage = False):
    device = '/gpu:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'
    with tf.device(device):
        hub_path = os.path.join(foundation_model_base,arch)

        module_keras=hub.KerasLayer(hub_path, trainable=True)
        model = tf.keras.Sequential(module_keras)
        model.build(input_shape=(None, 224,224,3))
        #model.add(Flatten()) # avrpool
        if not noAverage:
            model.add(AveragePooling2D(pool_size=(7, 7))) #TODO change to trainable conv layer
            model.add(Flatten())
        if not no_linear:
            model.add(Dense(1024, activation= None,use_bias=False)) # input_shape=(131072,)

    return model


class Identity(tf.keras.layers.Layer):
    def call(self, x):
        return x
    
class SimCLR(tf.keras.Model):
    def __init__(self, linear_eval=False, arch='path-50x1-remedis-s'):
        super().__init__()
        self.linear_eval = linear_eval
        self.arch = arch
        # resnet18 = models.resnet18(pretrained=False)
        remedis_model = foundationModel(arch = self.arch)
        remedis_model.fc = Identity()
        self.encoder = remedis_model
        self.projection = tf.keras.Sequential([
            tf.keras.layers.Dense(512),
            tf.keras.layers.ReLU(),
            tf.keras.layers.Dense(256)
        ])


    def call(self, x):
        if not self.linear_eval:
            x = tf.concat(x, axis=0)
        encoding = self.encoder(x)
        projection = self.projection(encoding)
        return projection



In [41]:

# Contrastive Loss

LABELS = tf.concat([tf.range(BATCH_SZ) for i in range(2)], axis=0)
LABELS = tf.cast(tf.equal(tf.expand_dims(LABELS, axis=0), tf.expand_dims(LABELS, axis=1)), tf.float32) #one-hot representations
LABELS = tf.cast(LABELS, tf.float32)

def ntxent_loss(features, temp):
    """
    NT-Xent Loss.

    Args:
        z1: The learned representations from first branch of projection head
        z2: The learned representations from second branch of projection head
    Returns:
        Loss
    """
    similarity_matrix = tf.matmul(features, features, transpose_b=True)
    mask = tf.eye(tf.shape(LABELS)[0], dtype=tf.bool)
    labels = tf.reshape(LABELS[~mask], (tf.shape(LABELS)[0], -1))
    similarity_matrix = tf.reshape(similarity_matrix[~mask], (tf.shape(similarity_matrix)[0], -1))

    positives = tf.reshape(similarity_matrix[tf.cast(labels, tf.bool)], (tf.shape(labels)[0], -1))

    negatives = tf.reshape(similarity_matrix[~tf.cast(labels, tf.bool)], (tf.shape(similarity_matrix)[0], -1))

    logits = tf.concat([positives, negatives], axis=1)
    labels = tf.zeros(tf.shape(logits)[0], dtype=tf.int64)

    logits = logits / temp
    return logits, labels


    #train


In [42]:
simclr_model = SimCLR()
criterion = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()

In [9]:
simclr_model.build(input_shape=(None, 224,224,3))
simclr_model.summary()


Model: "sim_clr"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential (Sequential)     (None, 1024)              25593408  
                                                                 
 sequential_1 (Sequential)   (None, 256)               656128    
                                                                 
Total params: 26,249,536
Trainable params: 26,249,536
Non-trainable params: 0
_________________________________________________________________


In [44]:
epochs = 1000
loss_values = []
# with tqdm(total=epochs) as pbar:
for epoch in range(epochs):
    # t0 = time.time()
    running_loss = 0.0
    for i, views in enumerate(deneme_ds):
        print("epoch"+str(epoch))
        print(views.shape)
        with tf.GradientTape() as tape:
            with tf.device(DEVICE):
                projections = simclr_model([view for view in views])
                logits, labels = ntxent_loss(projections, temp=2)
                loss = criterion(logits, labels)
        gradients = tape.gradient(loss, simclr_model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, simclr_model.trainable_variables))

        # print stats
        running_loss += loss.numpy()
        loss_values.append(loss.numpy())
        # if i % 10 == 9: # print every 10 mini-batches
        print(f"Epoch: {epoch+1} Batch: {i+1} Loss: {(running_loss/100):.4f}")
        running_loss = 0.0
        # pbar.update(1)
        # print(f"Time taken: {((time.time()-t0)/60):.3f} mins")

epoch0
(1, 224, 224, 3)


ValueError: Exception encountered when calling layer 'keras_layer_5' (type KerasLayer).

Could not find matching concrete function to call loaded from the SavedModel. Got:
  Positional arguments (1 total):
    * <tf.Tensor 'x:0' shape=(224, 224, 3) dtype=float32>
  Keyword arguments: {}

 Expected these arguments to match one of the following 2 option(s):

Option 1:
  Positional arguments (1 total):
    * TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_1')
  Keyword arguments: {}

Option 2:
  Positional arguments (1 total):
    * TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='x')
  Keyword arguments: {}

Call arguments received by layer 'keras_layer_5' (type KerasLayer):
  • inputs=tf.Tensor(shape=(224, 224, 3), dtype=float32)
  • training=None

In [19]:

# Assuming deneme_ds is already defined as in your provided code
batch_index = 0  # You can change this index to fetch different batches
batch = deneme_ds[batch_index]

# Check the size of the batch
batch_size = len(batch)
print(f"Batch size: {batch_size}")

# If you want to check the shape of individual images in the batch
image_shape = batch[0].shape if batch_size > 0 else None
print(f"Image shape: {image_shape}")

Batch size: 3
Image shape: (224, 224)


In [ ]:
# img = cv2.imread(tif_files[4])

# img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
transforms = A.Compose([
        A.RandomCrop(width=224, height=224),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.2, p=0.8),
        # A.ToGray(p=0.2),
        A.GaussianBlur(blur_limit=(3, 7), p=0.5),
        #ToTensorV2(),
    ])
 

# Apply the transformation
img_cropped = transforms(image=img)['image']
# img_cropped = img_cropped.transpose(2, 0, 1)
plt.imshow(img_cropped)

In [ ]:
img = cv2.imread(tif_files[4])
base_transforms = get_complete_transform(output_shape=out_shape, s=1.0)
img_trans = base_transforms(image=img)
plt.imshow(img_cropped)

In [ ]:
plt.plot(loss_values)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()